# Importing libraries and loading data

In [1]:
#Import libraries and load data
import pandas as pd
import numpy as np

In [2]:
transactions = pd.read_csv('trans_labelled_clean.csv')
cards = pd.read_csv('cards_data_south_africa.csv')
users = pd.read_csv('user_data_south_africa.csv')

trans_merged = pd.merge(transactions, cards, left_on='card_id', right_on='id')
trans_merged = pd.merge(trans_merged, users, left_on='client_id_x', right_on='id')
trans_merged.drop(columns=['id_y', 'client_id_y', 'id'])
trans_merged = trans_merged.rename(columns={
    'id_x': 'id',
    'client_id_x': 'user_id',
})

In [3]:
trans_merged['description'].unique()

array(['Miscellaneous Food Stores', 'Department Stores', 'Money Transfer',
       'Drinking Places (Alcoholic Beverages)', 'Book Stores',
       'Tolls and Bridge Fees', 'Grocery Stores, Supermarkets',
       'Taxicabs and Limousines', 'Service Stations',
       'Automotive Service Shops', 'Package Stores, Beer, Wine, Liquor',
       'Fast Food Restaurants', 'Discount Stores',
       'Telecommunication Services',
       'Utilities - Electric, Gas, Water, Sanitary',
       'Eating Places and Restaurants',
       'Betting (including Lottery Tickets, Casinos)', 'Wholesale Clubs',
       'Lodging - Hotels, Motels, Resorts',
       'Books, Periodicals, Newspapers', 'Drug Stores and Pharmacies',
       'Detective Agencies, Security Services', 'Family Clothing Stores',
       'Computer Network Services', 'Theatrical Producers',
       'Digital Goods - Media, Books, Apps', 'Recreational Sports, Clubs',
       'Sports Apparel, Riding Apparel Stores', 'Electronics Stores',
       'Athletic Field

# Amount Feature

In [4]:
trans_merged['is_negative_amount'] = trans_merged['amount'] < 0
trans_merged['abs_amount'] = trans_merged['amount'].abs()
trans_merged['is_negative_amount'] = trans_merged['is_negative_amount'].astype(int)
trans_merged['is_fraud'] = trans_merged['is_fraud'].astype(int)

In [5]:
if "amount" in trans_merged.columns:
    del trans_merged['amount']
trans_merged.head()

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,birth_month,gender,address,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,is_negative_amount,abs_amount
0,7475327,2022-02-19 05:51:55,1556,2972,swipe transaction,59935,kimberley,northern cape,8300.0,5499.0,...,7,Female,"405 Pretoria Street, Johannesburg",426222,868986,1982754,740,4,1,1386.00
1,7475328,2023-01-13 02:58:58,561,4575,swipe transaction,67570,pietermaritzburg,kwazulu-natal,3200.0,5311.0,...,6,Male,"374 Voortrekker Road, Bloemfontein",325368,663354,2018502,834,5,0,262.26
2,7475329,2024-07-03 23:41:24,1129,102,swipe transaction,27092,port elizabeth,eastern cape,6000.0,4829.0,...,4,Male,"146 Voortrekker Road, Gqeberha",304092,620082,657720,686,3,0,1440.00
3,7475332,2023-08-26 10:05:48,848,3915,swipe transaction,13051,polokwane,limpopo,700.0,5813.0,...,5,Male,"944 Church Street, East London",603522,1230516,1731276,711,2,0,835.38
4,7475333,2022-05-17 22:31:08,1807,165,swipe transaction,20519,bloemfontein,free state,9300.0,5942.0,...,12,Female,"658 Durban Road, Pietermaritzburg",459666,937170,1775034,828,5,0,86.58


In [6]:
trans_merged.columns

Index(['id', 'date', 'user_id', 'card_id', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud',
       'description', 'id_y', 'client_id_y', 'card_brand', 'card_type',
       'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed',
       'card_on_dark_web', 'id', 'current_age', 'retirement_age', 'birth_year',
       'birth_month', 'gender', 'address', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards',
       'is_negative_amount', 'abs_amount'],
      dtype='object')

# Temporal Features

In [7]:
df = trans_merged

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

# Basic time features
df['transaction_hour'] = df['date'].dt.hour
df['transaction_day'] = df['date'].dt.day
df['transaction_dayofweek'] = df['date'].dt.dayofweek
df['transaction_week'] = df['date'].dt.isocalendar().week
df['is_weekend'] = df['transaction_dayofweek'].isin([5, 6]).astype(int)
df['is_night'] = ((df['transaction_hour'] >= 22) | (df['transaction_hour'] <= 6)).astype(int)
df['is_rush_hour'] = ((df['transaction_hour'] >= 7) & (df['transaction_hour'] <= 9)) | \
                     ((df['transaction_hour'] >= 17) & (df['transaction_hour'] <= 19))

# User Behavioral Features

In [8]:
# User transaction patterns (need to sort by date first)
df = df.sort_values(['user_id', 'date'])

# Time since last transaction
df['time_since_last_txn'] = df.groupby('user_id')['date'].diff().dt.total_seconds() / 3600

# Amount statistics for user
df['user_avg_amount'] = df.groupby('user_id')['abs_amount'].transform(lambda x: x.expanding().mean().shift())
df['user_std_amount'] = df.groupby('user_id')['abs_amount'].transform(lambda x: x.expanding().std().shift())
df['amount_to_avg_ratio'] = df['abs_amount'] / (df['user_avg_amount'] + 1e-6)

In [9]:
# 0. Make sure ordering is temporal per user
df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

# 2. Transaction frequency – 24h window
txn_count_24h = (
    df.groupby('user_id')
      .apply(
          lambda g: (
              g.set_index('date')['abs_amount']
               .rolling('24h')        # time-based window
               .count()
          )
      )
      .reset_index(level=0, drop=True)
)

df['txn_count_24h'] = txn_count_24h.values  # assign by row order

# 3. Transaction frequency – 7d window
txn_count_7d = (
    df.groupby('user_id')
      .apply(
          lambda g: (
              g.set_index('date')['abs_amount']
               .rolling('7D')         # 7-day window
               .count()
          )
      )
      .reset_index(level=0, drop=True)
)

df['txn_count_7d'] = txn_count_7d.values

C:\Users\Geeks2_PC10\AppData\Local\Temp\ipykernel_14744\1402956000.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\Geeks2_PC10\AppData\Local\Temp\ipykernel_14744\1402956000.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [10]:
df.head(10)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,transaction_week,is_weekend,is_night,is_rush_hour,time_since_last_txn,user_avg_amount,user_std_amount,amount_to_avg_ratio,txn_count_24h,txn_count_7d
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,52,1,1,False,NaN,NaN,NaN,NaN,1.0,1.0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,52,1,1,False,4.281944,179.2800,NaN,31.927711,2.0,2.0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,52,1,1,False,1.028889,2951.6400,3920.709112,0.334431,3.0,3.0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,52,1,0,True,1.750556,2296.8000,2995.400849,0.121552,4.0,4.0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,52,1,0,False,6.293056,1792.3950,2645.621877,0.077327,5.0,5.0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,52,1,0,False,0.654167,1461.6360,2407.590895,0.793330,6.0,6.0
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,52,1,0,True,3.428333,1411.2900,2156.943080,0.129073,7.0,7.0
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,52,1,0,False,1.809444,1235.7000,2023.073358,5.629133,8.0,8.0
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,52,1,1,False,2.554444,1950.7275,2756.492521,0.074003,9.0,9.0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,52,1,1,False,6.496389,1750.0200,2647.833292,1.762643,8.0,10.0


In [11]:
pretty_cols = [
    'time_since_last_txn', 'user_avg_amount', 'user_std_amount',
    'amount_to_avg_ratio'
]

df[pretty_cols] = df[pretty_cols].round(3)

In [12]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,transaction_week,is_weekend,is_night,is_rush_hour,time_since_last_txn,user_avg_amount,user_std_amount,amount_to_avg_ratio,txn_count_24h,txn_count_7d
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,52,1,1,False,NaN,NaN,NaN,NaN,1.0,1.0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,52,1,1,False,4.282,179.280,NaN,31.928,2.0,2.0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,52,1,1,False,1.029,2951.640,3920.709,0.334,3.0,3.0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,52,1,0,True,1.751,2296.800,2995.401,0.122,4.0,4.0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,52,1,0,False,6.293,1792.395,2645.622,0.077,5.0,5.0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,52,1,0,False,0.654,1461.636,2407.591,0.793,6.0,6.0
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,52,1,0,True,3.428,1411.290,2156.943,0.129,7.0,7.0
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,52,1,0,False,1.809,1235.700,2023.073,5.629,8.0,8.0
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,52,1,1,False,2.554,1950.728,2756.493,0.074,9.0,9.0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,52,1,1,False,6.496,1750.020,2647.833,1.763,8.0,10.0


In [13]:
df['time_since_last_txn'] = df['time_since_last_txn'].fillna(9999)

In [14]:
df['user_avg_amount'] = df['user_avg_amount'].fillna(df['abs_amount'])
df['user_std_amount'] = df['user_std_amount'].fillna(0)

In [15]:
df.head(10)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,transaction_week,is_weekend,is_night,is_rush_hour,time_since_last_txn,user_avg_amount,user_std_amount,amount_to_avg_ratio,txn_count_24h,txn_count_7d
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,52,1,1,False,9999.000,179.280,0.000,NaN,1.0,1.0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,52,1,1,False,4.282,179.280,0.000,31.928,2.0,2.0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,52,1,1,False,1.029,2951.640,3920.709,0.334,3.0,3.0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,52,1,0,True,1.751,2296.800,2995.401,0.122,4.0,4.0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,52,1,0,False,6.293,1792.395,2645.622,0.077,5.0,5.0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,52,1,0,False,0.654,1461.636,2407.591,0.793,6.0,6.0
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,52,1,0,True,3.428,1411.290,2156.943,0.129,7.0,7.0
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,52,1,0,False,1.809,1235.700,2023.073,5.629,8.0,8.0
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,52,1,1,False,2.554,1950.728,2756.493,0.074,9.0,9.0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,52,1,1,False,6.496,1750.020,2647.833,1.763,8.0,10.0


# Merchant Risk Features

In [16]:
# Merchant risk score (historical fraud rate)
merchant_fraud_rate = df.groupby('merchant_id')['is_fraud'].transform(lambda x: x.expanding().mean().shift())
df['merchant_risk_score'] = merchant_fraud_rate.fillna(df['is_fraud'].mean())

# Merchant category risk
mcc_fraud_rate = df.groupby('mcc')['is_fraud'].transform(lambda x: x.expanding().mean().shift())
df['mcc_risk_score'] = mcc_fraud_rate.fillna(df['is_fraud'].mean())

In [17]:
df['amount_to_avg_ratio'] = df['amount_to_avg_ratio'].fillna(1.0)

In [18]:
# --- High-risk fraud category mapping from Transaction Description ---
# Assumptions:
# - Your column with the category text is called 'transaction_description'
# - The values look like the array you shared (e.g., 'Money Transfer', 'Fast Food Restaurants', etc.)
# - You use pandas as pd

import pandas as pd
import numpy as np

# 1) Normalize helper
def _norm(s):
    if pd.isna(s):
        return ""
    return " ".join(str(s).strip().lower().split())

# 2) Define risk buckets (normalized strings)
HIGH_RISK = {
    _norm(x) for x in [
        "Money Transfer",
        "Betting (including Lottery Tickets, Casinos)",
        "Package Stores, Beer, Wine, Liquor",
        "Drinking Places (Alcoholic Beverages)",
        "Digital Goods - Media, Books, Apps",
        "Digital Goods - Games",
        "Telecommunication Services",
        "Cable, Satellite, and Other Pay Television Services",
        "Computer Network Services",
        "Travel Agencies",  # often abused for account testing/refunds
        "Taxicabs and Limousines",  # card-present fraud patterns
        "Lodging - Hotels, Motels, Resorts",  # common for testing/cash-out
    ]
}

MEDIUM_RISK = {
    _norm(x) for x in [
        # General retail & fast-moving consumer goods
        "Department Stores",
        "Discount Stores",
        "Wholesale Clubs",
        "Family Clothing Stores",
        "Women's Ready-To-Wear Stores",
        "Shoe Stores",
        "Gift, Card, Novelty Stores",
        "Electronics Stores",
        "Sporting Goods Stores",
        "Sports Apparel, Riding Apparel Stores",
        "Household Appliance Stores",
        "Furniture, Home Furnishings, and Equipment Stores",
        "Floor Covering Stores",
        "Miscellaneous Home Furnishing Stores",
        "Hardware Stores",
        "Lawn and Garden Supply Stores",
        "Gardening Supplies",
        "Florists Supplies, Nursery Stock and Flowers",
        "Artist Supply Stores, Craft Shops",
        "Antique Shops",
        "Leather Goods",
        "Books, Periodicals, Newspapers",
        "Book Stores",
        "Music Stores - Musical Instruments",
        "Computers, Computer Peripheral Equipment",
        "Electronics Stores",

        # Food & restaurants
        "Fast Food Restaurants",
        "Eating Places and Restaurants",
        "Miscellaneous Food Stores",
        "Grocery Stores, Supermarkets",

        # Travel & transport (moderate risk, lots of disputes but also legit)
        "Airlines",
        "Cruise Lines",
        "Passenger Railways",
        "Railroad Passenger Transport",
        "Bus Lines",
        "Local and Suburban Commuter Transportation",
        "Tolls and Bridge Fees",
        "Service Stations",
        "Automotive Service Shops",
        "Automotive Parts and Accessories Stores",
        "Automotive Body Repair Shops",
        "Car Washes",
        "Towing Services",

        # Services where disputes happen but core business is legit
        "Cleaning and Maintenance Services",
        "Laundry Services",
        "Beauty and Barber Shops",
        "Cosmetic Stores",
        "Upholstery and Drapery Stores",
        "Postal Services - Government Only",
        "Travel Agencies",
        "Legal Services and Attorneys",
        "Accounting, Auditing, and Bookkeeping Services",
        "Tax Preparation Services",
        "Detective Agencies, Security Services",
        "Medical Services",
        "Doctors, Physicians",
        "Dentists and Orthodontists",
        "Podiatrists",
        "Chiropractors",
        "Hospitals",
        "Insurance Sales, Underwriting",

        # Entertainment / recreation
        "Motion Picture Theaters",
        "Theatrical Producers",
        "Recreational Sports, Clubs",
        "Athletic Fields, Commercial Sports",
        "Amusement Parks, Carnivals, Circuses",
    ]
}

# Everything else will default to LOW_RISK unless you later promote it

def map_risk_category(desc: str) -> str:
    d = _norm(desc)
    if d in HIGH_RISK:
        return "High"
    if d in MEDIUM_RISK:
        return "Medium"
    risky_keywords = [
        "money transfer", "remittance", "betting", "casino", "lottery",
        "liquor", "alcohol", "digital goods", "topup", "airtime"
    ]
    if any(k in d for k in risky_keywords):
        return "High"
    return "Low"

def map_risk_score(cat: str) -> int:
    return {"High": 3, "Medium": 2, "Low": 1}.get(cat, 1)

def add_high_risk_features(df: pd.DataFrame, col: str = "transaction_description") -> pd.DataFrame:
    out = df.copy()
    out["_desc_norm"] = out[col].apply(_norm)
    out["risk_category"] = out[col].apply(map_risk_category)
    out["risk_score"] = out["risk_category"].apply(map_risk_score).astype(np.int8)
    out["is_high_risk"] = (out["risk_category"] == "High").astype(np.int8)

    # Diagnostics: which labels were unseen by our explicit sets?
    known = HIGH_RISK.union(MEDIUM_RISK)
    unseen = sorted(set(out["_desc_norm"].unique()) - known)
    if unseen:
        print(f"[INFO] {len(unseen)} unseen/low-risk labels (defaulted to 'Low'). "
              f"Consider reviewing a sample to reclassify if needed.")
        # Optional: show a few examples
        print(unseen[:15])

    # Clean up helper column
    out.drop(columns=["_desc_norm"], inplace=True)
    return out

In [19]:
df = add_high_risk_features(df, col="description")
df[["description", "risk_category", "risk_score", "is_high_risk"]].head()


[INFO] 32 unseen/low-risk labels (defaulted to 'Low'). Consider reviewing a sample to reclassify if needed.
['bolt, nut, screw, rivet manufacturing', 'brick, stone, and related materials', 'coated and laminated products', 'drug stores and pharmacies', 'electroplating, plating, polishing services', 'fabricated structural metal products', 'heat treating metal services', 'heating, plumbing, air conditioning contractors', 'industrial equipment and supplies', 'ironwork', 'lighting, fixtures, electrical supplies', 'lumber and building materials', 'miscellaneous fabricated metal products', 'miscellaneous machinery and parts manufacturing', 'miscellaneous metal fabrication']


,description,risk_category,risk_score,is_high_risk
0,Eating Places and Restaurants,Medium,2,0
1,"Lighting, Fixtures, Electrical Supplies",Low,1,0
2,Drinking Places (Alcoholic Beverages),High,3,1
3,Drug Stores and Pharmacies,Low,1,0
4,Fast Food Restaurants,Medium,2,0


# Card Features

In [20]:
# Card age
df['card_age_days'] = (df['date'] - pd.to_datetime(df['acct_open_date'])).dt.days

# Time since PIN change
df['years_since_pin_change'] = df['date'].dt.year - df['year_pin_last_changed']

C:\Users\Geeks2_PC10\AppData\Local\Temp\ipykernel_14744\1957612718.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['card_age_days'] = (df['date'] - pd.to_datetime(df['acct_open_date'])).dt.days


In [21]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,amount_to_avg_ratio,txn_count_24h,txn_count_7d,merchant_risk_score,mcc_risk_score,risk_category,risk_score,is_high_risk,card_age_days,years_since_pin_change
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,1.000,1.0,1.0,0.001488,0.001488,Medium,2,0,5236,8
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,31.928,2.0,2.0,0.001488,0.001488,Low,1,0,3987,11
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,0.334,3.0,3.0,0.001488,0.001488,High,3,1,3987,11
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,0.122,4.0,4.0,0.001488,0.001488,Low,1,0,5236,8
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,0.077,5.0,5.0,0.001488,0.001488,Medium,2,0,3987,11
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,0.793,6.0,6.0,0.000000,0.000000,High,3,1,5236,8
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,0.129,7.0,7.0,0.000000,0.000000,Medium,2,0,3987,11
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,5.629,8.0,8.0,0.001488,0.001488,Medium,2,0,5236,8
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,0.074,9.0,9.0,0.001488,0.000000,Medium,2,0,5236,8
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,1.763,8.0,10.0,0.001488,0.001488,Medium,2,0,5237,8


In [22]:
'''df = df.sort_values('date')

# Merchant-level cumulative stats
df['merchant_total_cum'] = (
    df.groupby('merchant_id').cumcount()
)

df['merchant_fraud_cum'] = (
    df.groupby('merchant_id')['is_fraud']
      .apply(lambda s: s.shift().fillna(0).cumsum())
      .reset_index(level=0, drop=True)
)

df['merchant_risk_score'] = (
    df['merchant_fraud_cum'] / df['merchant_total_cum'].replace(0, np.nan)
).fillna(0)'''

"df = df.sort_values('date')\n\n# Merchant-level cumulative stats\ndf['merchant_total_cum'] = (\n    df.groupby('merchant_id').cumcount()\n)\n\ndf['merchant_fraud_cum'] = (\n    df.groupby('merchant_id')['is_fraud']\n      .apply(lambda s: s.shift().fillna(0).cumsum())\n      .reset_index(level=0, drop=True)\n)\n\ndf['merchant_risk_score'] = (\n    df['merchant_fraud_cum'] / df['merchant_total_cum'].replace(0, np.nan)\n).fillna(0)"

In [23]:
'''df['mcc_total_cum'] = df.groupby('mcc').cumcount()

df['mcc_fraud_cum'] = (
    df.groupby('mcc')['is_fraud']
      .apply(lambda s: s.shift().fillna(0).cumsum())
      .reset_index(level=0, drop=True)
)

df['mcc_risk_score'] = (
    df['mcc_fraud_cum'] / df['mcc_total_cum'].replace(0, np.nan)
).fillna(0)'''

"df['mcc_total_cum'] = df.groupby('mcc').cumcount()\n\ndf['mcc_fraud_cum'] = (\n    df.groupby('mcc')['is_fraud']\n      .apply(lambda s: s.shift().fillna(0).cumsum())\n      .reset_index(level=0, drop=True)\n)\n\ndf['mcc_risk_score'] = (\n    df['mcc_fraud_cum'] / df['mcc_total_cum'].replace(0, np.nan)\n).fillna(0)"

# Location & Distance Features

In [24]:
'''# Extract user zip from address (if available)
df['user_zip'] = df['address'].str.extract(r'(\d{5})')

# Merchant state/city frequency (unusual locations)
df['merchant_state_freq'] = df.groupby('merchant_state')['merchant_state'].transform('count')
df['state_risk'] = df['merchant_state_freq'].rank(pct=True)  # Less frequent states might be riskier

# Transaction distance (if you have user location)
# df['distance_from_home'] = haversine(user_lat, user_lon, merchant_lat, merchant_lon)'''

<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Geeks2_PC10\AppData\Local\Temp\ipykernel_14744\3011362600.py:2: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df['user_zip'] = df['address'].str.extract(r'(\d{5})')


"# Extract user zip from address (if available)\ndf['user_zip'] = df['address'].str.extract(r'(\\d{5})')\n\n# Merchant state/city frequency (unusual locations)\ndf['merchant_state_freq'] = df.groupby('merchant_state')['merchant_state'].transform('count')\ndf['state_risk'] = df['merchant_state_freq'].rank(pct=True)  # Less frequent states might be riskier\n\n# Transaction distance (if you have user location)\n# df['distance_from_home'] = haversine(user_lat, user_lon, merchant_lat, merchant_lon)"

# Interaction & Anomaly Features

In [25]:
# High-value transaction indicator
df['is_high_value'] = (df['abs_amount'] > df['abs_amount'].quantile(0.95)).astype(int)

# Card brand risk interaction
brand_risk = df.groupby('card_brand')['is_fraud'].mean()
df['card_brand_risk'] = df['card_brand'].map(brand_risk)

# MCC amount anomaly
mcc_avg_amount = df.groupby('mcc')['abs_amount'].mean()
df['mcc_amount_deviation'] = (df['abs_amount'] - df['mcc'].map(mcc_avg_amount)) / (df['mcc'].map(mcc_avg_amount) + 1e-6)

In [26]:
# Compute per-user mean & std directly on df (no merge)
df['user_hour_mean'] = df.groupby('user_id')['transaction_hour'].transform('mean')
df['user_hour_std']  = df.groupby('user_id')['transaction_hour'].transform('std')

# Handle zero/NaN std (users with 1 transaction)
df['user_hour_std'] = df['user_hour_std'].fillna(0)

# Compute unusual hour
df['unusual_hour'] = (
    (df['transaction_hour'] - df['user_hour_mean']).abs() >
    (2 * df['user_hour_std'])
).astype('int8')

In [27]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,risk_score,is_high_risk,card_age_days,years_since_pin_change,is_high_value,card_brand_risk,mcc_amount_deviation,user_hour_mean,user_hour_std,unusual_hour
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,2,0,5236,8,0,0.001461,-0.622306,11.494377,6.927022,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,1,0,3987,11,1,0.001461,0.196647,11.494377,6.927022,0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,3,1,3987,11,0,0.001461,1.174011,11.494377,6.927022,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,1,0,5236,8,0,0.001461,-0.658544,11.494377,6.927022,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,2,0,3987,11,0,0.001461,-0.707022,11.494377,6.927022,0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,3,1,5236,8,0,0.001461,1.553789,11.494377,6.927022,0
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,2,0,3987,11,0,0.001461,-0.616239,11.494377,6.927022,0
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,2,0,5236,8,1,0.001461,8.286222,11.494377,6.927022,0
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,2,0,5236,8,0,0.001461,-0.695873,11.494377,6.927022,0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,2,0,5237,8,1,0.001461,2.007727,11.494377,6.927022,0


In [28]:
cols_to_drop = ["meand_x", "std_x"]

existing_cols = [col for col in cols_to_drop if col in df.columns]
df.drop(columns=existing_cols, inplace=True)

In [29]:
df.drop(columns=df.columns.intersection(["meand_x", "std_x"]), inplace=True)

In [30]:
df.drop(columns=['mean_x', 'std_x'], errors='ignore')

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,risk_score,is_high_risk,card_age_days,years_since_pin_change,is_high_value,card_brand_risk,mcc_amount_deviation,user_hour_mean,user_hour_std,unusual_hour
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,2,0,5236,8,0,0.001461,-0.622306,11.494377,6.927022,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,1,0,3987,11,1,0.001461,0.196647,11.494377,6.927022,0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,3,1,3987,11,0,0.001461,1.174011,11.494377,6.927022,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,1,0,5236,8,0,0.001461,-0.658544,11.494377,6.927022,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,2,0,3987,11,0,0.001461,-0.707022,11.494377,6.927022,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8615528,13254825,2024-12-28 08:20:17,1998,2160,swipe transaction,27715,bloemfontein,free state,9300.0,5300.0,...,2,0,6512,17,0,0.001490,-0.945475,11.308337,6.959943,0
8615529,21588267,2024-12-28 10:31:43,1998,2160,swipe transaction,45756,pietermaritzburg,kwazulu-natal,3200.0,7538.0,...,2,0,6512,17,0,0.001490,-0.801909,11.308337,6.959943,0
8615530,10035325,2024-12-28 13:33:34,1998,2160,swipe transaction,13646,east london,eastern cape,5200.0,7538.0,...,2,0,6512,17,0,0.001490,-0.804596,11.308337,6.959943,0
8615531,10404322,2024-12-28 14:43:12,1998,1258,swipe transaction,46284,bloemfontein,free state,9300.0,5411.0,...,2,0,5444,13,0,0.001461,-0.675154,11.308337,6.959943,0


In [31]:
user_hour_robust = (
    df.groupby('user_id')['transaction_hour']
      .agg(
          hour_median='median',
          mad=lambda x: np.median(np.abs(x - np.median(x)))
      )
      .reset_index()
)

df = df.merge(user_hour_robust, on='user_id', how='left')

In [32]:
df['robust_z_hour'] = (
    (df['transaction_hour'] - df['hour_median']) / (1.4826 * df['mad'])
)

In [33]:
df.loc[
    (df['is_fraud'] == 1) & (df['robust_z_hour'].abs() >= 1),
    ['robust_z_hour', 'is_fraud']
].head(200)

,robust_z_hour,is_fraud
1371,-1.348982,1
2602,1.236566,1
13022,1.011736,1
13764,-1.236566,1
19266,-1.348982,1
...,...,...
451768,-1.011736,1
451816,1.124151,1
451918,1.124151,1
452744,-1.011736,1


In [34]:
user_iqr = (
    df.groupby('user_id')['transaction_hour']
      .quantile([0.25, 0.75])
      .unstack()
      .reset_index()
      .rename(columns={0.25: 'q1', 0.75: 'q3'})
)

user_iqr['iqr'] = user_iqr['q3'] - user_iqr['q1']

In [35]:
df = df.merge(user_iqr, on='user_id', how='left')
K = 1.5

df['lower_bound'] = df['q1'] - K * df['iqr']
df['upper_bound'] = df['q3'] + K * df['iqr']

In [36]:
df['unusual_hour_flag'] = (
    (df['transaction_hour'] < df['lower_bound']) |
    (df['transaction_hour'] > df['upper_bound'])
).astype(int)

In [37]:
df['unusual_hour_flag'] = np.where(
    df['iqr'] == 0,
    (df['transaction_hour'] != df['q1']).astype(int),
    df['unusual_hour_flag']
)

In [38]:
user_txn_count = df.groupby('user_id').size()
df['user_txn_count'] = df['user_id'].map(user_txn_count)

df['unusual_hour_flag'] = np.where(
    df['user_txn_count'] < 10,
    0,
    df['unusual_hour_flag']
)

In [33]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,hour_median,mad,robust_z_hour,q1,q3,iqr,lower_bound,upper_bound,unusual_hour_flag,user_txn_count
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,12.0,6.0,-1.348982,5.0,18.0,13.0,-14.5,37.5,0,8358
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,12.0,6.0,-0.786906,5.0,18.0,13.0,-14.5,37.5,0,8358
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,12.0,6.0,-0.674491,5.0,18.0,13.0,-14.5,37.5,0,8358
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,12.0,6.0,-0.562076,5.0,18.0,13.0,-14.5,37.5,0,8358
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,12.0,6.0,0.224830,5.0,18.0,13.0,-14.5,37.5,0,8358
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,12.0,6.0,0.224830,5.0,18.0,13.0,-14.5,37.5,0,8358
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,12.0,6.0,0.674491,5.0,18.0,13.0,-14.5,37.5,0,8358
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,12.0,6.0,0.899321,5.0,18.0,13.0,-14.5,37.5,0,8358
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,12.0,6.0,1.124151,5.0,18.0,13.0,-14.5,37.5,0,8358
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,12.0,6.0,-0.786906,5.0,18.0,13.0,-14.5,37.5,0,8358


In [34]:
df.columns

Index(['id', 'date', 'user_id', 'card_id', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud',
       'description', 'card_brand', 'card_type', 'card_number', 'expires',
       'cvv', 'has_chip', 'num_cards_issued', 'credit_limit', 'acct_open_date',
       'year_pin_last_changed', 'card_on_dark_web', 'current_age',
       'retirement_age', 'birth_year', 'birth_month', 'gender', 'address',
       'per_capita_income', 'yearly_income', 'total_debt', 'credit_score',
       'num_credit_cards', 'is_negative_amount', 'abs_amount',
       'transaction_hour', 'transaction_day', 'transaction_dayofweek',
       'transaction_week', 'is_weekend', 'is_night', 'is_rush_hour',
       'time_since_last_txn', 'user_avg_amount', 'user_std_amount',
       'amount_to_avg_ratio', 'txn_count_24h', 'txn_count_7d',
       'merchant_risk_score', 'mcc_risk_score', 'card_age_days',
       'years_since_pin_change', 'is_high_value', 'card_brand_risk',
       'mcc_

In [39]:
cols_to_drop = ["mcc_risk_score", "merchant_risk_score"]

existing_cols = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols)

In [ ]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,hour_median,mad,robust_z_hour,q1,q3,iqr,lower_bound,upper_bound,unusual_hour_flag,user_txn_count
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,12.0,6.0,-1.348982,5.0,18.0,13.0,-14.5,37.5,0,8358
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,12.0,6.0,-0.786906,5.0,18.0,13.0,-14.5,37.5,0,8358
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,12.0,6.0,-0.674491,5.0,18.0,13.0,-14.5,37.5,0,8358
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,12.0,6.0,-0.562076,5.0,18.0,13.0,-14.5,37.5,0,8358
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,12.0,6.0,0.224830,5.0,18.0,13.0,-14.5,37.5,0,8358
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,12.0,6.0,0.224830,5.0,18.0,13.0,-14.5,37.5,0,8358
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,12.0,6.0,0.674491,5.0,18.0,13.0,-14.5,37.5,0,8358
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,12.0,6.0,0.899321,5.0,18.0,13.0,-14.5,37.5,0,8358
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,12.0,6.0,1.124151,5.0,18.0,13.0,-14.5,37.5,0,8358
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,12.0,6.0,-0.786906,5.0,18.0,13.0,-14.5,37.5,0,8358


In [37]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,hour_median,mad,robust_z_hour,q1,q3,iqr,lower_bound,upper_bound,unusual_hour_flag,user_txn_count
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,12.0,6.0,-1.348982,5.0,18.0,13.0,-14.5,37.5,0,8358
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,12.0,6.0,-0.786906,5.0,18.0,13.0,-14.5,37.5,0,8358
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,12.0,6.0,-0.674491,5.0,18.0,13.0,-14.5,37.5,0,8358
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,12.0,6.0,-0.562076,5.0,18.0,13.0,-14.5,37.5,0,8358
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,12.0,6.0,0.224830,5.0,18.0,13.0,-14.5,37.5,0,8358
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,12.0,6.0,0.224830,5.0,18.0,13.0,-14.5,37.5,0,8358
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,12.0,6.0,0.674491,5.0,18.0,13.0,-14.5,37.5,0,8358
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,12.0,6.0,0.899321,5.0,18.0,13.0,-14.5,37.5,0,8358
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,12.0,6.0,1.124151,5.0,18.0,13.0,-14.5,37.5,0,8358
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,12.0,6.0,-0.786906,5.0,18.0,13.0,-14.5,37.5,0,8358


In [40]:
def iqr_bounds(s: pd.Series, k: float = 1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

In [41]:
# Compute user-specific Q1 and Q3
q1 = df.groupby('user_id')['abs_amount'].transform(lambda x: x.quantile(0.25))
q3 = df.groupby('user_id')['abs_amount'].transform(lambda x: x.quantile(0.75))

iqr = q3 - q1

# Per-user lower and upper bounds
lower_iqr = q1 - 1.5 * iqr
upper_iqr = q3 + 1.5 * iqr

In [42]:
df['is_iqr_outlier'] = (
    (df['abs_amount'] < lower_iqr) |
    (df['abs_amount'] > upper_iqr)
).astype(int)

In [43]:
df_iqr_outliers = df.loc[
    df['is_iqr_outlier'] == 1,
    ['user_id', 'abs_amount', 'date']
].sort_values(['user_id', 'abs_amount'])

In [44]:
df['amount_zscore'] = (
    (df['abs_amount'] - df.groupby('user_id')['abs_amount'].transform('mean')) /
    (df.groupby('user_id')['abs_amount'].transform('std') + 1e-6)
)
df['is_z_outlier'] = (df['amount_zscore'] > 2.5).astype(int)

In [43]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,q1,q3,iqr,lower_bound,upper_bound,unusual_hour_flag,user_txn_count,is_iqr_outlier,amount_zscore,is_z_outlier
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,-0.653388,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,1,3.213431,1
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,-0.090010,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,-0.583719,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,-0.681757,0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0.030247,0
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,-0.651379,0
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,1,4.072556,1
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,-0.677740,0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,1,1.372788,0


In [45]:
cols_to_drop = ["mcc_amount_deviation", "amount_zscore", "card_brand_risk"]

existing_cols = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols)

In [45]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,robust_z_hour,q1,q3,iqr,lower_bound,upper_bound,unusual_hour_flag,user_txn_count,is_iqr_outlier,is_z_outlier
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,-1.348982,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,-0.786906,5.0,18.0,13.0,-14.5,37.5,0,8358,1,1
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,-0.674491,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,-0.562076,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,0.224830,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,0.224830,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,0.674491,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,0.899321,5.0,18.0,13.0,-14.5,37.5,0,8358,1,1
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,1.124151,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,-0.786906,5.0,18.0,13.0,-14.5,37.5,0,8358,1,0


In [46]:
df["yearly_income"] = df["yearly_income"].replace(0, np.nan)
df["amt_to_income_ratio"] = df["abs_amount"] / df["yearly_income"]
df["amt_to_income_ratio"] = df["amt_to_income_ratio"].fillna(0)

In [47]:
df["amt_to_income_ratio"] = df["abs_amount"] / df["yearly_income"]

In [48]:
df["log_amt_to_income_ratio"] = np.log1p(df["amt_to_income_ratio"])

In [49]:
df["income_ratio_outlier"] = (df["amt_to_income_ratio"] > 0.1).astype(int)

In [50]:
df["yearly_income"] = df["yearly_income"].replace(0, np.nan)
df["amt_to_income_ratio"] = (df["abs_amount"] / df["yearly_income"]).fillna(0)

In [44]:
df.head()

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,iqr,lower_bound,upper_bound,unusual_hour_flag,user_txn_count,is_iqr_outlier,is_z_outlier,amt_to_income_ratio,log_amt_to_income_ratio,income_ratio_outlier
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,13.0,-14.5,37.5,0,8358,0,0,0.000167,0.000167,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,13.0,-14.5,37.5,0,8358,1,1,0.005334,0.005320,0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,13.0,-14.5,37.5,0,8358,0,0,0.000920,0.000920,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,13.0,-14.5,37.5,0,8358,0,0,0.000260,0.000260,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,13.0,-14.5,37.5,0,8358,0,0,0.000129,0.000129,0


In [45]:
df['income_ratio_outlier'].value_counts()

income_ratio_outlier
0    8595249
1      20284
Name: count, dtype: int64

In [51]:
cols_to_drop = ["amt_to_income_ratio", "log_amt_to_income_ratio"]

existing_cols = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols)

In [47]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,q1,q3,iqr,lower_bound,upper_bound,unusual_hour_flag,user_txn_count,is_iqr_outlier,is_z_outlier,income_ratio_outlier
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,1,1,0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0,0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0,0
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0,0
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,1,1,0
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,0,0,0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,5.0,18.0,13.0,-14.5,37.5,0,8358,1,0,0


In [52]:
def build_time_window_features(
    df,
    ts_col="date",
    amount_col="abs_amount"
):
    # ---- mandatory ordering ----
    df = df.sort_values(["user_id", ts_col]).reset_index(drop=True)

    # ============================
    # USER-LEVEL FEATURES
    # ============================
    g_user = df.groupby("user_id", group_keys=False)

    df["user_txn_count_24h"] = (
        g_user.rolling("24h", on=ts_col)[amount_col]
        .count()
        .shift(1)
        .reset_index(drop=True)
    )

    df["user_txn_count_7d"] = (
        g_user.rolling("7d", on=ts_col)[amount_col]
        .count()
        .shift(1)
        .reset_index(drop=True)
    )

    df["user_amt_sum_24h"] = (
        g_user.rolling("24h", on=ts_col)[amount_col]
        .sum()
        .shift(1)
        .reset_index(drop=True)
    )

    # ---- stronger than raw sums ----
    df["user_avg_amt_24h"] = (
        df["user_amt_sum_24h"] /
        (df["user_txn_count_24h"] + 1)
    )

    # ============================
    # CLEANUP
    # ============================
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    return df

In [53]:
#Applied
df = build_time_window_features(
    df,
    ts_col="date",
    amount_col="abs_amount"
)

In [57]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,upper_bound,unusual_hour_flag,user_txn_count,is_iqr_outlier,is_z_outlier,income_ratio_outlier,user_txn_count_24h,user_txn_count_7d,user_amt_sum_24h,user_avg_amt_24h
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,37.5,0,8358,0,0,0,NaN,NaN,NaN,NaN
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,37.5,0,8358,1,1,0,1.0,1.0,179.28,89.640000
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,37.5,0,8358,0,0,0,2.0,2.0,5903.28,1967.760000
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,37.5,0,8358,0,0,0,3.0,3.0,6890.40,1722.600000
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,37.5,0,8358,0,0,0,4.0,4.0,7169.58,1433.916000
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,37.5,0,8358,0,0,0,5.0,5.0,7308.18,1218.030000
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,37.5,0,8358,0,0,0,6.0,6.0,8467.74,1209.677143
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,37.5,0,8358,1,1,0,7.0,7.0,8649.90,1081.237500
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,37.5,0,8358,0,0,0,8.0,8.0,15605.82,1733.980000
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,37.5,0,8358,1,0,0,9.0,9.0,15750.18,1575.018000


In [54]:
df['user_txn_count_24h'] = df['user_txn_count_24h'].fillna(0).astype('Int64')
df['user_txn_count_7d']  = df['user_txn_count_7d'].fillna(0).astype('Int64')
df['user_amt_sum_24h'] = df['user_amt_sum_24h'].fillna(0)
df['user_avg_amt_24h'] = df['user_avg_amt_24h'].fillna(df['abs_amount'])
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,upper_bound,unusual_hour_flag,user_txn_count,is_iqr_outlier,is_z_outlier,income_ratio_outlier,user_txn_count_24h,user_txn_count_7d,user_amt_sum_24h,user_avg_amt_24h
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,37.5,0,8358,0,0,0,0,0,0.00,179.280000
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,37.5,0,8358,1,1,0,1,1,179.28,89.640000
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,37.5,0,8358,0,0,0,2,2,5903.28,1967.760000
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,37.5,0,8358,0,0,0,3,3,6890.40,1722.600000
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,37.5,0,8358,0,0,0,4,4,7169.58,1433.916000
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,37.5,0,8358,0,0,0,5,5,7308.18,1218.030000
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,37.5,0,8358,0,0,0,6,6,8467.74,1209.677143
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,37.5,0,8358,1,1,0,7,7,8649.90,1081.237500
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,37.5,0,8358,0,0,0,8,8,15605.82,1733.980000
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,37.5,0,8358,1,0,0,9,9,15750.18,1575.018000


In [55]:
# Use this - it's safe and simple
df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

# Merchant features
df['prev_visits_to_merchant'] = (
    df.groupby(['user_id', 'merchant_id']).cumcount().shift(1).fillna(0)
)
df['is_first_merchant_visit'] = (df['prev_visits_to_merchant'] == 0).astype(int)

# City features  
df['prev_visits_to_city'] = (
    df.groupby(['user_id', 'merchant_city']).cumcount().shift(1).fillna(0)
)
df['is_first_city_visit'] = (df['prev_visits_to_city'] == 0).astype(int)

# Optional: Add a "rare visit" flag (visited less than 3 times before)
df['is_rare_merchant'] = (df['prev_visits_to_merchant'] < 3).astype(int)

In [60]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,income_ratio_outlier,user_txn_count_24h,user_txn_count_7d,user_amt_sum_24h,user_avg_amt_24h,prev_visits_to_merchant,is_first_merchant_visit,prev_visits_to_city,is_first_city_visit,is_rare_merchant
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,0,0,0,0.00,179.280000,0.0,1,0.0,1,1
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,0,1,1,179.28,89.640000,0.0,1,0.0,1,1
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,0,2,2,5903.28,1967.760000,0.0,1,0.0,1,1
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,0,3,3,6890.40,1722.600000,0.0,1,1.0,0,1
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,0,4,4,7169.58,1433.916000,0.0,1,0.0,1,1
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,0,5,5,7308.18,1218.030000,0.0,1,0.0,1,1
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,0,6,6,8467.74,1209.677143,1.0,0,1.0,0,1
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,0,7,7,8649.90,1081.237500,1.0,0,0.0,1,1
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,0,8,8,15605.82,1733.980000,0.0,1,0.0,1,1
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,0,9,9,15750.18,1575.018000,0.0,1,2.0,0,1


In [54]:
df['is_rare_merchant'].value_counts()

is_rare_merchant
0    7975965
1     639568
Name: count, dtype: int64

In [56]:
cols_to_drop = ["prev_visits_to_merchant", "prev_visits_to_city"]

existing_cols = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols)

In [54]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,is_iqr_outlier,is_z_outlier,income_ratio_outlier,user_txn_count_24h,user_txn_count_7d,user_amt_sum_24h,user_avg_amt_24h,is_first_merchant_visit,is_first_city_visit,is_rare_merchant
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,0,0,0,0,0,0.00,179.280000,1,1,1
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,1,1,0,1,1,179.28,89.640000,1,1,1
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,0,0,0,2,2,5903.28,1967.760000,1,1,1
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,0,0,0,3,3,6890.40,1722.600000,1,0,1
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,0,0,0,4,4,7169.58,1433.916000,1,1,1
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,0,0,0,5,5,7308.18,1218.030000,1,1,1
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,0,0,0,6,6,8467.74,1209.677143,0,0,1
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,1,1,0,7,7,8649.90,1081.237500,0,1,1
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,0,0,0,8,8,15605.82,1733.980000,1,1,1
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,1,0,0,9,9,15750.18,1575.018000,1,0,1


In [57]:
df['user_city'] = df['address'].str.split(',').str[1].str.strip()

city_to_region = {
    "cape town": "western cape",
    "pretoria": "gauteng",
    "johannesburg": "gauteng",
    "durban": "kwazulu-natal",
    "pietermaritzburg": "kwazulu-natal",
    "gqeberha": "eastern cape",
    "east london": "eastern cape",
    "bloemfontein": "free state",
}

# Normalization
def normalize_city(s):
    if pd.isna(s): return None
    return (str(s).strip().lower()
            .replace('.', '')
            .replace('-', ' ')
            .replace(',', ''))

# Apply
df['user_city'] = df['user_city'].map(normalize_city)
df['user_region'] = df['user_city'].map(city_to_region).fillna("unknown")

In [56]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,income_ratio_outlier,user_txn_count_24h,user_txn_count_7d,user_amt_sum_24h,user_avg_amt_24h,is_first_merchant_visit,is_first_city_visit,is_rare_merchant,user_city,user_region
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,0,0,0,0.00,179.280000,1,1,1,bloemfontein,free state
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,0,1,1,179.28,89.640000,1,1,1,bloemfontein,free state
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,0,2,2,5903.28,1967.760000,1,1,1,bloemfontein,free state
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,0,3,3,6890.40,1722.600000,1,0,1,bloemfontein,free state
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,0,4,4,7169.58,1433.916000,1,1,1,bloemfontein,free state
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,0,5,5,7308.18,1218.030000,1,1,1,bloemfontein,free state
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,0,6,6,8467.74,1209.677143,0,0,1,bloemfontein,free state
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,0,7,7,8649.90,1081.237500,0,1,1,bloemfontein,free state
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,0,8,8,15605.82,1733.980000,1,1,1,bloemfontein,free state
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,0,9,9,15750.18,1575.018000,1,0,1,bloemfontein,free state


In [58]:
# Clean and standardize text first
df['user_city'] = df['user_city'].str.lower().str.strip().fillna('')
df['merchant_city'] = df['merchant_city'].str.lower().str.strip().fillna('')
df['user_region'] = df['user_region'].str.lower().str.strip().fillna('')
df['merchant_state'] = df['merchant_state'].str.lower().str.strip().fillna('')

# Now create features
df["is_same_city"] = (
    (df['user_city'] != '') &
    (df['merchant_city'] != '') &
    (df['user_city'] == df['merchant_city'])
).astype(int)

df["is_same_region"] = (
    (df['user_region'] != '') &
    (df['merchant_state'] != '') &
    (df['user_region'] == df['merchant_state'])
).astype(int)

# Simplify different region - no need for AND with is_same_region
df["is_different_region"] = (
    (df['user_region'] != '') &
    (df['merchant_state'] != '') &
    (df['user_region'] != df['merchant_state'])
).astype(int)

In [67]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,user_amt_sum_24h,user_avg_amt_24h,is_first_merchant_visit,is_first_city_visit,is_rare_merchant,user_city,user_region,is_same_city,is_same_region,is_different_region
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,0.00,179.280000,1,1,1,bloemfontein,free state,0,0,1
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,179.28,89.640000,1,1,1,bloemfontein,free state,0,0,1
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,5903.28,1967.760000,1,1,1,bloemfontein,free state,0,0,1
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,6890.40,1722.600000,1,0,1,bloemfontein,free state,0,0,1
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,7169.58,1433.916000,1,1,1,bloemfontein,free state,0,0,1
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,7308.18,1218.030000,1,1,1,bloemfontein,free state,0,0,1
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,8467.74,1209.677143,0,0,1,bloemfontein,free state,0,0,1
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,8649.90,1081.237500,0,1,1,bloemfontein,free state,0,0,1
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,15605.82,1733.980000,1,1,1,bloemfontein,free state,0,0,1
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,15750.18,1575.018000,1,0,1,bloemfontein,free state,0,0,1


In [58]:
df['is_same_region'].value_counts()

is_same_region
0    7501495
1    1114038
Name: count, dtype: int64

In [59]:
if 'is_different_region' in df.columns:
    del df['is_different_region']

In [ ]:
'''# 1. Day of week (0=Monday to 6=Sunday)
df["dow_sin"] = np.sin(2 * np.pi * df["transaction_dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["transaction_dayofweek"] / 7)

# 2. Day of month (1-31, though month length varies)
df["day_of_month"] = df["date"].dt.day
df["dom_sin"] = np.sin(2 * np.pi * df["day_of_month"] / 31)
df["dom_cos"] = np.cos(2 * np.pi * df["day_of_month"] / 31)

# 3. Month of year (1-12)
df["month"] = df["date"].dt.month
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# This is exactly right
df["hour_sin"] = np.sin(2 * np.pi * df["transaction_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["transaction_hour"] / 24)'''

In [56]:
df.head(20)

,id,date,user_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,...,prev_visits_to_merchant,is_first_merchant_visit,prev_visits_to_city,is_first_city_visit,is_rare_merchant,user_city,user_region,is_same_city,is_same_region,is_different_region
0,8927991,2022-01-01 00:46:02,0,4639,179.28,swipe transaction,55060,polokwane,limpopo,700.0,...,0.0,1,0.0,1,1,bloemfontein,free state,0,0,1
1,20704571,2022-01-01 05:02:57,0,1271,5724.00,chip transaction,3558,cape town,western cape,8000.0,...,0.0,1,0.0,1,1,bloemfontein,free state,0,0,1
2,18886224,2022-01-01 06:04:41,0,1271,987.12,chip transaction,32164,cape town,western cape,8000.0,...,0.0,1,0.0,1,1,bloemfontein,free state,0,0,1
3,10496251,2022-01-01 07:49:43,0,4639,279.18,swipe transaction,60510,durban,kwazulu-natal,4000.0,...,0.0,1,1.0,0,1,bloemfontein,free state,0,0,1
4,21259165,2022-01-01 14:07:18,0,1271,138.60,chip transaction,98648,east london,eastern cape,5200.0,...,0.0,1,0.0,1,1,bloemfontein,free state,0,0,1
5,22576560,2022-01-01 14:46:33,0,4639,1159.56,chip transaction,32164,polokwane,limpopo,700.0,...,0.0,1,0.0,1,1,bloemfontein,free state,0,0,1
6,14689091,2022-01-01 18:12:15,0,1271,182.16,swipe transaction,55060,rustenburg,north west,300.0,...,1.0,0,1.0,0,1,bloemfontein,free state,0,0,1
7,13965939,2022-01-01 20:00:49,0,4639,6955.92,swipe transaction,41527,nelspruit,mpumalanga,1200.0,...,1.0,0,0.0,1,1,bloemfontein,free state,0,0,1
8,15116306,2022-01-01 22:34:05,0,4639,144.36,swipe transaction,13153,cape town,western cape,8000.0,...,0.0,1,0.0,1,1,bloemfontein,free state,0,0,1
9,7652484,2022-01-02 05:03:52,0,4639,3084.66,swipe transaction,90865,cape town,western cape,8000.0,...,0.0,1,2.0,0,1,bloemfontein,free state,0,0,1


In [60]:
df = df.sort_values(["user_id", "date"])
df["prev_city"] = df.groupby("user_id")["merchant_city"].shift(1)
df["city_changed"] = (
    (df["merchant_city"] != df["prev_city"]) &
    df["prev_city"].notna()
).astype(int)

In [61]:
df["time_diff_sec"] = (
    (df["date"] - df.groupby("user_id")["date"].shift(1))
    .dt.total_seconds()
)

In [62]:
df["city_change_fast"] = (
    (df["city_changed"] == 1) &
    (df["time_diff_sec"].between(60, 3600))
).astype(int)

In [ ]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,is_rare_merchant,user_city,user_region,is_same_city,is_same_region,prev_city,city_changed,time_diff_hours,time_diff_sec,city_change_fast
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,1,bloemfontein,free state,0,0,NaN,0,NaN,NaN,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,1,bloemfontein,free state,0,0,polokwane,1,15415.0,15415.0,0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,1,bloemfontein,free state,0,0,cape town,0,3704.0,3704.0,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,1,bloemfontein,free state,0,0,cape town,1,6302.0,6302.0,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,1,bloemfontein,free state,0,0,durban,1,22655.0,22655.0,0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,1,bloemfontein,free state,0,0,east london,1,2355.0,2355.0,1
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,1,bloemfontein,free state,0,0,polokwane,1,12342.0,12342.0,0
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,1,bloemfontein,free state,0,0,rustenburg,1,6514.0,6514.0,0
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,1,bloemfontein,free state,0,0,nelspruit,1,9196.0,9196.0,0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,1,bloemfontein,free state,0,0,cape town,0,23387.0,23387.0,0


In [63]:
cols_to_drop = ["time_diff_hours", "time_diff_sec", "prev_city"]

existing_cols = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols)

In [64]:
df.columns

Index(['id', 'date', 'user_id', 'card_id', 'use_chip', 'merchant_id',
       'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors', 'is_fraud',
       'description', 'id_y', 'client_id_y', 'card_brand', 'card_type',
       'card_number', 'expires', 'cvv', 'has_chip', 'num_cards_issued',
       'credit_limit', 'acct_open_date', 'year_pin_last_changed',
       'card_on_dark_web', 'id', 'current_age', 'retirement_age', 'birth_year',
       'birth_month', 'gender', 'address', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards',
       'is_negative_amount', 'abs_amount', 'transaction_hour',
       'transaction_day', 'transaction_dayofweek', 'transaction_week',
       'is_weekend', 'is_night', 'is_rush_hour', 'time_since_last_txn',
       'user_avg_amount', 'user_std_amount', 'amount_to_avg_ratio',
       'txn_count_24h', 'txn_count_7d', 'card_age_days',
       'years_since_pin_change', 'is_high_value', 'user_hour_mean',
       'user_hour_std', '

In [64]:
cols_to_drop = ["transaction_day", "transaction_week", "transaction_week", 
                "years_since_pin_change", "mad", "q1", "q3", "iqr", "lower_bound", "upper_bound",
                  "user_txn_count"]

existing_cols = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols)

In [66]:
df.head(20)

,id,date,user_id,card_id,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,...,user_avg_amt_24h,is_first_merchant_visit,is_first_city_visit,is_rare_merchant,user_city,user_region,is_same_city,is_same_region,city_changed,city_change_fast
0,8927991,2022-01-01 00:46:02,0,4639,swipe transaction,55060,polokwane,limpopo,700.0,5812.0,...,179.280000,1,1,1,bloemfontein,free state,0,0,0,0
1,20704571,2022-01-01 05:02:57,0,1271,chip transaction,3558,cape town,western cape,8000.0,3640.0,...,89.640000,1,1,1,bloemfontein,free state,0,0,1,0
2,18886224,2022-01-01 06:04:41,0,1271,chip transaction,32164,cape town,western cape,8000.0,5813.0,...,1967.760000,1,1,1,bloemfontein,free state,0,0,0,0
3,10496251,2022-01-01 07:49:43,0,4639,swipe transaction,60510,durban,kwazulu-natal,4000.0,5912.0,...,1722.600000,1,0,1,bloemfontein,free state,0,0,1,0
4,21259165,2022-01-01 14:07:18,0,1271,chip transaction,98648,east london,eastern cape,5200.0,5814.0,...,1433.916000,1,1,1,bloemfontein,free state,0,0,1,0
5,22576560,2022-01-01 14:46:33,0,4639,chip transaction,32164,polokwane,limpopo,700.0,5813.0,...,1218.030000,1,1,1,bloemfontein,free state,0,0,1,1
6,14689091,2022-01-01 18:12:15,0,1271,swipe transaction,55060,rustenburg,north west,300.0,5812.0,...,1209.677143,0,0,1,bloemfontein,free state,0,0,1,0
7,13965939,2022-01-01 20:00:49,0,4639,swipe transaction,41527,nelspruit,mpumalanga,1200.0,5310.0,...,1081.237500,0,1,1,bloemfontein,free state,0,0,1,0
8,15116306,2022-01-01 22:34:05,0,4639,swipe transaction,13153,cape town,western cape,8000.0,5812.0,...,1733.980000,1,1,1,bloemfontein,free state,0,0,1,0
9,7652484,2022-01-02 05:03:52,0,4639,swipe transaction,90865,cape town,western cape,8000.0,5311.0,...,1575.018000,1,0,1,bloemfontein,free state,0,0,0,0


In [65]:
df.to_csv('transactions_feat.csv', index=False)